# Methods S3: Prediction of protein biochemical properties

In [2]:
!pip3 install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 25.0 MB/s eta 0:00:00


In [ ]:
from Bio.SeqUtils import molecular_weight
from Bio.SeqUtils import IsoelectricPoint as IP
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import numpy as np
import pandas as pd

# unit testing

In [ ]:
# Peptide sequence
sequence = "MKPFDSVTGVVFPGPKNRLFLWTFLVIVSLISAAYFLGNAFSAKLEYKQRLARWGLIYSMPPDTNSNACKVQCRPSGTQPLPQGILATTSNLETRPLWDSPVNNNLRISTRPLNLLAMAVGVKQKEVVDKIVKKFPSTDFVVMLFHYDGFVDGWKNLTWSNRAIHVSAINQTKWWFAKRFLHPDIVADYNYIFLWDEDLLVDNFNPKRYLSIIKEEGLEISQPALDRGKSEIHHPLTAHIAGSKVHRRYYKLKGSGRCDDSSTAPPCLGWVEMMAPVFSKKSWQCVWHMIQNDLIHAWGLDRQLGYCAQGDRMRNVGVVDSEYIVHLGLPTLGGSSSNGNEARDFIFLC"

# Function to calculate molecular weight
def calculate_molecular_weight(seq):
    return molecular_weight(seq, seq_type='protein')

# Function to calculate theoretical pI
def calculate_pI(seq):
    ip = IP.IsoelectricPoint(seq)
    return ip.pi()

def calculate_instability_index(seq):
    analyzed_seq = ProteinAnalysis(seq)
    return analyzed_seq.instability_index()

def calculate_aliphatic_index(seq):
    # Count aliphatic residues (A, V, I, L)
    count_A = seq.count('A')
    count_V = seq.count('V')
    count_I = seq.count('I')
    count_L = seq.count('L')
    total_length = len(seq)

    # Calculate mole fractions
    x_A = count_A / total_length
    x_V = count_V / total_length
    x_IL = (count_I + count_L) / total_length

    # Aliphatic index formula: X(Ala) + 2.9 × X(Val) + 3.9 × (X(Ile) + X(Leu))
    aliphatic_index = x_A + 2.9 * x_V + 3.9 * x_IL
    return aliphatic_index * 100

def calculate_gravy(seq):
    # Hydropathy scale values for amino acids (Kyte-Doolittle)
    hydropathy_scale = {
        'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4,
        'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8,
        'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
    }
    return np.mean([hydropathy_scale[aa] for aa in seq])

def calculate_total_atoms(seq):
    # Atomic composition per amino acid (C, H, N, O, S) in the peptide chain
    # These are the counts for each AA in the middle of a chain (accounting for peptide bonds)
    amino_acid_compositions = {
        'A': (3, 5, 1, 1, 0),  # Ala: C3H5NO
        'R': (6, 12, 4, 1, 0),  # Arg: C6H12N4O
        'N': (4, 6, 2, 2, 0),   # Asn: C4H6N2O2
        'D': (4, 5, 1, 3, 0),   # Asp: C4H5NO3
        'C': (3, 5, 1, 1, 1),   # Cys: C3H5NOS
        'Q': (5, 8, 2, 2, 0),   # Gln: C5H8N2O2
        'E': (5, 7, 1, 3, 0),   # Glu: C5H7NO3
        'G': (2, 3, 1, 1, 0),   # Gly: C2H3NO
        'H': (6, 7, 3, 1, 0),   # His: C6H7N3O
        'I': (6, 11, 1, 1, 0),  # Ile: C6H11NO
        'L': (6, 11, 1, 1, 0),  # Leu: C6H11NO
        'K': (6, 12, 2, 1, 0),  # Lys: C6H12N2O
        'M': (5, 9, 1, 1, 1),   # Met: C5H9NOS
        'F': (9, 9, 1, 1, 0),   # Phe: C9H9NO
        'P': (5, 7, 1, 1, 0),   # Pro: C5H7NO
        'S': (3, 5, 1, 2, 0),   # Ser: C3H5NO2
        'T': (4, 7, 1, 2, 0),   # Thr: C4H7NO2
        'W': (11, 10, 2, 1, 0), # Trp: C11H10N2O
        'Y': (9, 9, 1, 2, 0),   # Tyr: C9H9NO2
        'V': (5, 9, 1, 1, 0)    # Val: C5H9NO
    }

    total_atoms = np.sum([np.array(amino_acid_compositions[aa]) for aa in seq], axis=0)

    total_atoms += np.array([0, 2, 0, 1, 0])

    return total_atoms

# Calculate total amino acids
total_amino_acids = len(sequence)

# Results
molecular_weight_result = calculate_molecular_weight(sequence)
pI_result = calculate_pI(sequence)
instability_index_result = calculate_instability_index(sequence)
aliphatic_index_result = calculate_aliphatic_index(sequence)
gravy_result = calculate_gravy(sequence)
total_atoms_result = calculate_total_atoms(sequence)

# Output results
print(f"Total Amino Acids: {total_amino_acids}")
print(f"Molecular Weight: {molecular_weight_result:.4f} Da")
print(f"Theoretical pI: {pI_result:.2f}")
print(f"Instability Index: {instability_index_result:.2f}")
print(f"Aliphatic Index: {aliphatic_index_result:.2f}")
print(f"GRAVY: {gravy_result:.4f}")  # Positive = hydrophobic, negative = hydrophilic
print(f"Total Atoms (C, H, N, O, S): {total_atoms_result}")

Total Amino Acids: 349
Molecular Weight: 39609.1893 Da
Theoretical pI: 9.06
Instability Index: 36.68
Aliphatic Index: 87.97
GRAVY: -0.1705
Total Atoms (C, H, N, O, S): [1802 2768  486  493   15]


# File

In [ ]:
def calculate_molecular_weight(seq):
    try:
        return molecular_weight(seq, seq_type='protein')
    except:
        return None

def calculate_pI(seq):
    try:
        ip = IP.IsoelectricPoint(seq)
        return ip.pi()
    except:
        return None

def calculate_instability_index(seq):
    try:
        analyzed_seq = ProteinAnalysis(seq)
        return analyzed_seq.instability_index()
    except:
        return None

def calculate_aliphatic_index(seq):
    try:
        count_A = seq.count('A')
        count_V = seq.count('V')
        count_I = seq.count('I')
        count_L = seq.count('L')
        total_length = len(seq)
        x_A = count_A / total_length
        x_V = count_V / total_length
        x_IL = (count_I + count_L) / total_length
        aliphatic_index = x_A + 2.9 * x_V + 3.9 * x_IL
        return aliphatic_index * 100
    except:
        return None

def calculate_gravy(seq):
    try:
        hydropathy_scale = {
            'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4,
            'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8,
            'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
        }
        return np.mean([hydropathy_scale[aa] for aa in seq])
    except:
        return None

def calculate_total_atoms(seq):
    try:
        amino_acid_compositions = {
            'A': (3, 5, 1, 1, 0), 'R': (6, 12, 4, 1, 0), 'N': (4, 6, 2, 2, 0), 'D': (4, 5, 1, 3, 0),
            'C': (3, 5, 1, 1, 1), 'Q': (5, 8, 2, 2, 0), 'E': (5, 7, 1, 3, 0), 'G': (2, 3, 1, 1, 0),
            'H': (6, 7, 3, 1, 0), 'I': (6, 11, 1, 1, 0), 'L': (6, 11, 1, 1, 0), 'K': (6, 12, 2, 1, 0),
            'M': (5, 9, 1, 1, 1), 'F': (9, 9, 1, 1, 0), 'P': (5, 7, 1, 1, 0), 'S': (3, 5, 1, 2, 0),
            'T': (4, 7, 1, 2, 0), 'W': (11, 10, 2, 1, 0), 'Y': (9, 9, 1, 2, 0), 'V': (5, 9, 1, 1, 0)
        }
        total_atoms = np.sum([np.array(amino_acid_compositions[aa]) for aa in seq], axis=0)
        total_atoms += np.array([0, 2, 0, 1, 0])
        return total_atoms
    except:
        return [None, None, None, None, None]

def process_peptide_sequence(transcript_id, peptide):
    if pd.isna(peptide) or not isinstance(peptide, str) or len(peptide.strip()) == 0:
        return {
            'Transcript id': transcript_id,
            'Peptide': None,
            'Total Amino Acids': None,
            'Molecular Weight (Da)': None,
            'Theoretical pI': None,
            'Instability Index': None,
            'Aliphatic Index': None,
            'GRAVY': None,
            'Total C Atoms': None,
            'Total H Atoms': None,
            'Total N Atoms': None,
            'Total O Atoms': None,
            'Total S Atoms': None,
            'Status': 'Empty peptide sequence'
        }

    try:
        peptide = peptide.strip()
        if not all(aa in 'ACDEFGHIKLMNPQRSTVWY' for aa in peptide):
            raise ValueError("Contains invalid amino acid characters")

        results = {
            'Transcript id': transcript_id,
            'Peptide': peptide,
            'Total Amino Acids': len(peptide),
            'Molecular Weight (Da)': round(calculate_molecular_weight(peptide), 4),
            'Theoretical pI': round(calculate_pI(peptide), 2),
            'Instability Index': round(calculate_instability_index(peptide), 2),
            'Aliphatic Index': round(calculate_aliphatic_index(peptide), 2),
            'GRAVY': round(calculate_gravy(peptide), 4),
            'Status': 'Success'
        }

        atoms = calculate_total_atoms(peptide)
        results.update({
            'Total C Atoms': atoms[0],
            'Total H Atoms': atoms[1],
            'Total N Atoms': atoms[2],
            'Total O Atoms': atoms[3],
            'Total S Atoms': atoms[4]
        })

        return results

    except Exception as e:
        return {
            'Transcript id': transcript_id,
            'Peptide': peptide,
            'Total Amino Acids': None,
            'Molecular Weight (Da)': None,
            'Theoretical pI': None,
            'Instability Index': None,
            'Aliphatic Index': None,
            'GRAVY': None,
            'Total C Atoms': None,
            'Total H Atoms': None,
            'Total N Atoms': None,
            'Total O Atoms': None,
            'Total S Atoms': None,
            'Status': f'Error: {str(e)}'
        }

def process_excel_file(input_file, output_file):
    try:
        df = pd.read_excel(input_file)

        # Verify required columns exist
        if 'Transcript id' not in df.columns or 'Peptide Sequence' not in df.columns:
            raise ValueError("Input file must contain 'Transcript id' and 'Peptide' columns")

        results = []
        for _, row in df.iterrows():
            result = process_peptide_sequence(row['Transcript id'], row['Peptide Sequence'])
            results.append(result)

        results_df = pd.DataFrame(results)
        results_df.to_excel(output_file, index=False)
        print(f"Successfully processed {len(results)} sequences. Results saved to {output_file}")

    except Exception as e:
        print(f"Error processing file: {str(e)}")

# Example usage
input_filename = "Data S1.xlsx"
output_filename = "Data S7.xlsx"
process_excel_file(input_filename, output_filename)

Successfully processed 38818 sequences. Results saved to peptide_properties_results.xlsx
